# DSA 4060 – Practical Quiz: Personalized Movie Recommender
**Name:** Sundus Moheddin  
**Student ID:** 668726  
**Assigned User ID:** 27  (last two digits 26 → 26 MOD 40 = 26 → 26 + 1 = 27)

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STUDENT_ID = 668726
USER_ID = (STUDENT_ID % 100) % 40 + 1
print("Assigned User ID:", USER_ID)

Assigned User ID: 27


## Task 1 – Load and Inspect the Data

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
print("movies.csv – first five rows"); display(movies.head())
print("ratings.csv – first five rows"); display(ratings.head())

movies.csv – first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


ratings.csv – first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [3]:
print("movies.csv :", movies.shape[0], "rows,", movies.shape[1], "columns")
print("ratings.csv :", ratings.shape[0], "rows,", ratings.shape[1], "columns")

movies.csv : 36 rows, 3 columns
ratings.csv : 440 rows, 3 columns


In [4]:
print("Missing values in movies:\n", movies.isnull().sum(), "\n")
print("Missing values in ratings:\n", ratings.isnull().sum())

Missing values in movies:
 movie_id    0
title       0
genres      0
dtype: int64 

Missing values in ratings:
 user_id     0
movie_id    0
rating      0
dtype: int64


**Finding:** Neither dataset has any missing values (every column has 0 nulls), so no cleaning or imputation was needed. `movies.csv` is a catalogue of 36 movies with pipe-separated genres, and `ratings.csv` holds 440 ratings from 40 users.

## Task 2 – Profile the Assigned User

In [5]:
user = (ratings[ratings.user_id == USER_ID]
        .merge(movies, on="movie_id")
        .sort_values("rating", ascending=False)
        .reset_index(drop=True))
print(f"User {USER_ID} has rated {len(user)} movies (mean rating {user.rating.mean():.2f})")
user[["movie_id","title","genres","rating"]]

User 27 has rated 11 movies (mean rating 2.82)


,movie_id,title,genres,rating
0,23,Coco,Animation|Family,4.5
1,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.5
2,4,Arrival,Sci-Fi|Drama,3.0
3,24,Spider-Man: Into the Spider-Verse,Animation|Superhero,3.0
4,16,Crazy Rich Asians,Romance|Comedy,3.0
5,3,The Matrix,Sci-Fi|Action,2.5
6,2,Interstellar,Sci-Fi|Drama,2.5
7,28,Knives Out,Mystery|Comedy,2.5
8,10,Black Panther,Action|Superhero,2.5
9,5,Edge of Tomorrow,Sci-Fi|Action,2.0


In [6]:
top3 = user.nlargest(3, "rating")[["title","genres","rating"]]
top3

,title,genres,rating
0,Coco,Animation|Family,4.5
1,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.5
2,Arrival,Sci-Fi|Drama,3.0


In [7]:
# Average rating per genre for this user (evidence for preferences)
g = user.assign(genre=user.genres.str.split("|")).explode("genre")
genre_stats = g.groupby("genre").rating.agg(["mean","count"]).sort_values("mean", ascending=False)
genre_stats

,mean,count
genre,,
Family,4.500000,1
Animation,3.750000,2
Adventure,3.500000,1
Romance,3.000000,1
Comedy,3.000000,3
Superhero,2.750000,2
Mystery,2.500000,1
Sci-Fi,2.500000,4
Drama,2.500000,3


**Preferred genres:** The user's strongest preference is **Animation** (and the closely related **Family**). Coco (Animation|Family) is their highest rating at 4.5, and Animation has the best genre average (3.75 across Coco and Spider-Man: Into the Spider-Verse). Comedy is a weaker second preference: Jumanji (Comedy|Adventure) got 3.5 and the Comedy average is 3.0. By contrast, Sci-Fi, Drama and Action movies were all rated 2.0–3.0 (averages of about 2.3–2.5), so the user does not seem to enjoy them. The user is also a fairly harsh rater overall (mean about 2.8), and only one movie reaches 4.0, so Coco is the clearest signal of taste.

## Task 3 – Build the Recommendation Logic

**Step 1 – Movies the user likes (rating ≥ 4.0):**

In [8]:
liked = user[user.rating >= 4.0]
liked[["movie_id","title","genres","rating"]]

,movie_id,title,genres,rating
0,23,Coco,Animation|Family,4.5


**Step 2 – Convert genres into numerical features** with `CountVectorizer`, splitting on `|` so that multi-word/hyphenated genres such as `Sci-Fi` stay intact.

In [9]:
vec = CountVectorizer(tokenizer=lambda s: s.split("|"), token_pattern=None, lowercase=True)
genre_matrix = vec.fit_transform(movies.genres)
print("Genre vocabulary:", list(vec.get_feature_names_out()))
print("Feature matrix shape:", genre_matrix.shape)

Genre vocabulary: ['action', 'adventure', 'animation', 'biography', 'comedy', 'crime', 'drama', 'family', 'horror', 'musical', 'mystery', 'romance', 'sci-fi', 'sports', 'superhero', 'thriller']
Feature matrix shape: (36, 16)


**Step 3 – Cosine similarity** between every movie and each liked movie. A candidate's score is its mean similarity to the liked movies.

In [10]:
liked_idx = movies.index[movies.movie_id.isin(liked.movie_id)]
sim = cosine_similarity(genre_matrix, genre_matrix[liked_idx])
movies["sim_to_liked"] = sim.mean(axis=1)
movies["best_match"] = [liked.title.iloc[sim[i].argmax()] if sim[i].max() > 0 else None for i in range(len(movies))]
movies[["title","genres","sim_to_liked"]].sort_values("sim_to_liked", ascending=False).head(8)

,title,genres,sim_to_liked
22,Coco,Animation|Family,1.0
21,Finding Nemo,Animation|Adventure,0.5
23,Spider-Man: Into the Spider-Verse,Animation|Superhero,0.5
20,Toy Story,Animation|Comedy,0.5
3,Arrival,Sci-Fi|Drama,0.0
2,The Matrix,Sci-Fi|Action,0.0
1,Interstellar,Sci-Fi|Drama,0.0
0,Inception,Sci-Fi|Thriller,0.0


Only two unrated movies share a genre with Coco (Toy Story and Finding Nemo, both Animation), so the remaining candidates all have a similarity of 0. To still return exactly five movies in a sensible order, **ties are broken using the user's own average rating of each candidate's genres** (from the genre table in Task 2).

In [11]:
affinity = genre_stats["mean"].to_dict()
def genre_affinity(genres):
    vals = [affinity[x] for x in genres.split("|") if x in affinity]
    return sum(vals)/len(vals) if vals else 0
movies["affinity"] = movies.genres.apply(genre_affinity)

**Step 4 – Exclude already-rated movies and return exactly five.**

In [12]:
rated_ids = set(user.movie_id)
cands = movies[~movies.movie_id.isin(rated_ids)]
top5 = (cands.sort_values(["sim_to_liked","affinity"], ascending=False)
             .head(5).reset_index(drop=True))
assert len(top5) == 5 and not set(top5.movie_id) & rated_ids
print("Recommended titles:")
for t in top5.title: print(" -", t)

Recommended titles:
 - Finding Nemo
 - Toy Story
 - La La Land
 - The Hangover
 - Mad Max: Fury Road


## Task 4 – Explain the Recommendations

In [13]:
def reason(r):
    shared = set(r.genres.split("|")) & set(liked.genres.iloc[0].split("|"))
    if r.sim_to_liked > 0:
        return (f"Shares the {', '.join(shared)} genre with Coco (4.5, the user's only 4.0+ movie); "
                f"cosine similarity {r.sim_to_liked:.2f}.")
    gens = [x for x in r.genres.split("|") if x in affinity]
    detail = ", ".join(f"{x} avg {affinity[x]:.2f}" for x in gens)
    return (f"No genre overlap with Coco, but fits the user's other ratings ({detail}); "
            f"ranked by genre affinity {r.affinity:.2f}.")
results = pd.DataFrame({
    "Rank": range(1, 6),
    "Recommended Movie": top5.title,
    "Genres": top5.genres,
    "Reason": top5.apply(reason, axis=1)})
pd.set_option("display.max_colwidth", None)
results

,Rank,Recommended Movie,Genres,Reason
0,1,Finding Nemo,Animation|Adventure,"Shares the Animation genre with Coco (4.5, the user's only 4.0+ movie); cosine similarity 0.50."
1,2,Toy Story,Animation|Comedy,"Shares the Animation genre with Coco (4.5, the user's only 4.0+ movie); cosine similarity 0.50."
2,3,La La Land,Romance|Musical,"No genre overlap with Coco, but fits the user's other ratings (Romance avg 3.00); ranked by genre affinity 3.00."
3,4,The Hangover,Comedy,"No genre overlap with Coco, but fits the user's other ratings (Comedy avg 3.00); ranked by genre affinity 3.00."
4,5,Mad Max: Fury Road,Action|Adventure,"No genre overlap with Coco, but fits the user's other ratings (Action avg 2.33, Adventure avg 3.50); ranked by genre affinity 2.92."


## Task 5 – Critical Thinking

**Limitation:** This user has only one movie rated 4.0 or higher (Coco), so the whole profile rests on a single movie's genres. Only two unrated movies share a genre with it, and the other three recommendations had a similarity of 0 and had to be ranked by a fallback rule. Genre labels are also very coarse, so the recommender cannot tell why the user liked Coco (for example its story, music or tone).

**Improvement:** Make it a hybrid recommender: add collaborative filtering, finding the users whose ratings are most similar to User 27's and recommending movies those neighbours rated highly. This uses the other 39 users' ratings, so it would not depend on a single liked movie and would help when genre overlap runs out. Using a lower, user-specific "like" threshold (for example relative to this user's own mean rating) would also give more positive examples.